# Start by connecting to google drive and GitHub
We do this to avoid having to wait while our runtime-time is counting down.
pre-upload to google drive to save compute-time.

In [ ]:
from google.colab import drive
from google.colab import runtime
from google.colab import output
drive.mount('/content/drive')

In [ ]:
repo = 'https://github.com/Frederic-P/automotive_project.git'
!git clone {repo}

# Imports

In [ ]:
import pandas as pd
import os
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from automotive_project.utils import cnn_helpers
import json

zip to runtime

In [ ]:
datadir = '/content/drive/MyDrive/'
testdir = os.path.join(datadir, 'testdata')
traindir = os.path.join(datadir, 'traindata')
# depending on what data is uploaded to GDRIVE, this will choose a different angle to query against. 
#   Naming convention should be following: i.e.: put zips in: /content/drive/MyDrive/testdata/front.zip and /content/drive/MyDrive/traidnata/front.zip
#   this will traina model on pictures from the front. 
angle = os.listdir(traindir)[0].rstrip('.zip')  
model_dest_dir = os.path.join(datadir, 'model_output', 'angled_model')
os.makedirs(model_dest_dir, exist_ok=True)

gbase_test = '/content/testdata'
gbase_train = '/content/traindata'

!unzip {os.path.join(testdir, angle+'.zip')} -d {gbase_test}
!unzip {os.path.join(traindir, angle+'.zip')} -d {gbase_train}

In [ ]:
output.clear()  #allows me to reconnect, otherwise data is too slow or something https://github.com/googlecolab/colabtools/issues/2492

In [ ]:
columns = ['idx', 'brand', 'angle', 'path']
df_train = pd.read_csv(os.path.join(gbase_train, angle, 'datafile.csv'), skiprows=1, names = columns)
df_test = pd.read_csv(os.path.join(gbase_test, angle, 'datafile.csv'),  skiprows=1, names = columns)

In [ ]:
def set_paths(df):
  df = df.copy()
  df['abs_path'] = '/content'+df.path.str.split('Downloads').str[-1]
  df = df.drop(columns=['path'])
  return df

def fake_cols(df, fake_columns = ['yolobox_top_left_x', 'yolobox_top_left_y', 'yolobox_bottom_right_x', 'yolobox_bottom_right_y']):
  """resnet_learner expects yolobox coords in the code - will simply provide fake
  coordinates so it's happy without making breaking changes in the code. """
  df = df.copy()
  for col in fake_columns:
    df[col] = -1
  return df


In [ ]:
df_train = set_paths(df_train)
df_test = set_paths(df_test)


In [ ]:
SHAPE = 224   #required for resnet
BATCH_SIZE = 32     #how big ar teh batches for the online learning part
MAX_EPOCHS = 100    #upper limit of epochs per learning task -
                    #    NOTE THAT there is early stopping and lr plateau just as in nobteook 2.


## VERY IMPORTANT TO REALIZE THIS: we have pre-cropped the data on a physical system
# to use as little space as possible in the cloud so I could at least fit in the
# data for angle based models... because of that we do NOT need to crop again
# however, the outcome of this notebook would be the same as running it on the
# original data, on a bare metal GPU, with CROP = TRUE
CROP = False

In [ ]:
device = cnn_helpers.system_pick_device()


In [ ]:
#make an encoder:
#   LabelEncoder sorts alphabetically and since all classes are always in the traindata, these numbers
#   can be considered stable.
brands = df_train.brand.unique()
label_encoder = LabelEncoder()
label_encoder.fit(brands)

In [ ]:
#apply label encoding on train and test dataframes.
df_train['y_encoded'] = label_encoder.transform(df_train['brand'])
df_test['y_encoded'] = label_encoder.transform(df_test['brand'])

In [ ]:
df_train = cnn_helpers.shuffle_df(df_train)
df_test = cnn_helpers.shuffle_df(df_test)

In [ ]:
df_train.sample(10)

Start training: Google drive is just big enough to fit one (2 might just work) batch of train/test data. The runtime of notebooks is restricted to 4 hours anyway - so I'm not going to code this as a for-loop. There's simply no point for this.

In [ ]:
X_train, y_train = cnn_helpers.get_X_y(df_train, 'y_encoded', ['brand'])
X_test, y_test = cnn_helpers.get_X_y(df_test, 'y_encoded', ['brand'])

In [ ]:
X_train = fake_cols(X_train)
X_test = fake_cols(X_test)

In [ ]:
name = f'trained model for -RESNET50- {angle}.keras'
model = cnn_helpers.resnet_learner(X_train, X_test, y_train, y_test, SHAPE, CROP, model_dest_dir, BATCH_SIZE, MAX_EPOCHS)
model.save(os.path.join(model_dest_dir, name))

In [ ]:
def label_encoder_to_dict(label_encoder):
  """Converts a LabelEncoder into a dictionary with integer as key and text label as value.
  ARGUMENTS:
    label_encoder: The LabelEncoder object.

  RETURNS:
    A dictionary mapping integer encoded labels with text labels as values.
  """
  d =  dict(zip(label_encoder.transform(label_encoder.classes_), label_encoder.classes_))
  d = {int(k): v for k, v in d.items()}
  return d

label_values = label_encoder_to_dict(label_encoder)


In [ ]:
json.dump(label_values, open(os.path.join(model_dest_dir, f'label_values_{angle}.json'), 'w'))

In [ ]:
print('Notebook completed, closing runtime')
runtime.unassign()